# Fraud Detection — Data Exploration

Exploring `train_transaction.csv` and `train_identity.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = "../data"

## train_transaction.csv

In [ ]:
trn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
print(f"Shape: {trn.shape}")
trn.head()

In [ ]:
trn.info(verbose=False, memory_usage="deep")

In [ ]:
# Target distribution
fraud_counts = trn["isFraud"].value_counts()
print(fraud_counts)
print(f"\nFraud rate: {fraud_counts[1] / len(trn):.2%}")

fraud_counts.plot(kind="bar", title="isFraud distribution", rot=0)
plt.show()

In [ ]:
# Missing values (top 20 columns)
missing = trn.isnull().mean().sort_values(ascending=False)
print(missing[missing > 0].head(20))

In [ ]:
# TransactionAmt distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
trn["TransactionAmt"].plot(kind="hist", bins=100, ax=axes[0], title="TransactionAmt")
np.log1p(trn["TransactionAmt"]).plot(kind="hist", bins=100, ax=axes[1], title="log1p(TransactionAmt)")
plt.tight_layout()
plt.show()

In [ ]:
# Numeric summary for key columns
trn[["TransactionAmt", "TransactionDT"]].describe()

In [ ]:
# Top categorical columns
for col in ["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"]:
    if col in trn.columns:
        print(f"\n--- {col} ---")
        print(trn[col].value_counts().head(10))

## train_identity.csv

In [ ]:
idn = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
print(f"Shape: {idn.shape}")
idn.head()

In [ ]:
idn.info(verbose=False, memory_usage="deep")

In [ ]:
# Missing values
missing_idn = idn.isnull().mean().sort_values(ascending=False)
print(missing_idn[missing_idn > 0].head(20))

In [ ]:
# Key identity categorical columns
for col in ["DeviceType", "DeviceInfo", "id_12", "id_15", "id_16", "id_28", "id_29", "id_35", "id_36"]:
    if col in idn.columns:
        print(f"\n--- {col} ---")
        print(idn[col].value_counts().head(10))

## Merged view (transaction + identity)

In [ ]:
merged = trn.merge(idn, on="TransactionID", how="left")
print(f"Transactions total:          {len(trn):,}")
print(f"Transactions with identity:  {idn['TransactionID'].nunique():,}  ({idn['TransactionID'].nunique()/len(trn):.1%})")
print(f"Merged shape: {merged.shape}")

In [ ]:
# Fraud rate by device type
if "DeviceType" in merged.columns:
    print(merged.groupby("DeviceType")["isFraud"].agg(["mean", "count"]).rename(columns={"mean": "fraud_rate"}))

In [ ]:
# Fraud rate by ProductCD
print(merged.groupby("ProductCD")["isFraud"].agg(["mean", "count"]).rename(columns={"mean": "fraud_rate"}).sort_values("fraud_rate", ascending=False))